# Week 2: Data Engineering & Feature Tables

## Goals:
- Build feature tables ready for modeling
- Standardize column names across datasets
- Engineer key features
- Create per-event, per-order, and per-station feature sets
- Save to Parquet/CSV for fast re-use

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## 1. Load and Standardize Datasets

In [ ]:
# Load datasets
customer_orders = pd.read_csv('../../Order Picking Dataset from a Warehouse of a Footwear Manufacturing Company/Customer_Order.csv', sep=';')
picking_waves = pd.read_csv('../../Order Picking Dataset from a Warehouse of a Footwear Manufacturing Company/Picking_Wave.csv', sep=';')
products = pd.read_csv('../../Order Picking Dataset from a Warehouse of a Footwear Manufacturing Company/Product.csv', sep=';')

# Convert creationDate to datetime
customer_orders['creationDate'] = pd.to_datetime(customer_orders['creationDate'], format='%d/%m/%Y %H:%M')

print("Datasets loaded successfully!")

## 2. Standardize Column Names

In [ ]:
# Standardize column names for customer orders
customer_orders_standard = customer_orders.rename(columns={
    'codCustomer': 'customer_id',
    'orderNumber': 'order_id',
    'orderToCollect': 'order_line',
    'Reference': 'sku_id',
    'Size (US)': 'size_us',
    'quantity (units)': 'quantity',
    'creationDate': 'timestamp',
    'waveNumber': 'wave_id',
    'operator': 'operator_id'
})

# Standardize column names for picking waves
picking_waves_standard = picking_waves.rename(columns={
    'waveNumber': 'wave_id',
    'reference': 'sku_id',
    'Size (US)': 'size_us',
    'quantityToPick (units)': 'quantity_to_pick',
    'locations': 'location',
    'operator': 'operator_id'
})

# Standardize column names for products
products_standard = products.rename(columns={
    'Reference': 'sku_id',
    'ABCCOD': 'abc_code',
    'Sector': 'sector'
})

print("Column names standardized successfully!")
print("Customer Orders columns:", list(customer_orders_standard.columns))
print("Picking Waves columns:", list(picking_waves_standard.columns))
print("Products columns:", list(products_standard.columns))

## 3. Engineer Key Features

### Calculate delta_weight, residual, latency_sec, queue_depth, distance_walked

In [ ]:
# Merge customer orders with product information
customer_orders_enriched = customer_orders_standard.merge(
    products_standard, 
    on='sku_id', 
    how='left'
)

# Calculate order sequence for each customer
customer_orders_enriched = customer_orders_enriched.sort_values(['customer_id', 'timestamp'])
customer_orders_enriched['order_sequence'] = customer_orders_enriched.groupby('customer_id').cumcount() + 1

# Calculate time-based features
customer_orders_enriched = customer_orders_enriched.sort_values(['operator_id', 'timestamp'])
customer_orders_enriched['prev_timestamp'] = customer_orders_enriched.groupby('operator_id')['timestamp'].shift(1)
customer_orders_enriched['latency_sec'] = (customer_orders_enriched['timestamp'] - customer_orders_enriched['prev_timestamp']).dt.total_seconds()

# Calculate queue depth (number of orders in wave)
wave_order_counts = customer_orders_enriched.groupby('wave_id').size().reset_index(name='wave_order_count')
customer_orders_enriched = customer_orders_enriched.merge(wave_order_counts, on='wave_id', how='left')

# Calculate order size (number of lines per order)
order_line_counts = customer_orders_enriched.groupby('order_id').size().reset_index(name='order_size')
customer_orders_enriched = customer_orders_enriched.merge(order_line_counts, on='order_id', how='left')

# For residual calculation, we'll simulate based on expected vs actual quantity
# In a real scenario, this would come from tolerance rules or measurements
customer_orders_enriched['expected_quantity'] = customer_orders_enriched['quantity']  # Placeholder
customer_orders_enriched['actual_quantity'] = customer_orders_enriched['quantity']    # Placeholder
customer_orders_enriched['residual'] = customer_orders_enriched['actual_quantity'] - customer_orders_enriched['expected_quantity']

# For delta_weight, we'll simulate based on product characteristics
# In a real scenario, this would come from product weight data
customer_orders_enriched['unit_weight'] = np.random.uniform(0.1, 2.0, len(customer_orders_enriched))  # Simulated weight
customer_orders_enriched['delta_weight'] = customer_orders_enriched['quantity'] * customer_orders_enriched['unit_weight']

# For distance_walked, we'll simulate based on location changes
# In a real scenario, this would come from actual distance calculations between locations
customer_orders_enriched['distance_walked'] = np.random.uniform(10, 100, len(customer_orders_enriched))  # Simulated distance

print("Key features engineered successfully!")
print("New features added:")
new_features = ['order_sequence', 'prev_timestamp', 'latency_sec', 'wave_order_count', 'order_size', 
                'expected_quantity', 'actual_quantity', 'residual', 'unit_weight', 'delta_weight', 'distance_walked']
print(new_features)

In [ ]:
# Display the enriched dataset with new features
customer_orders_enriched.head()

## 4. Create Feature Tables

### Event-Level Features

In [ ]:
# Event-level features (each row represents a picking event)
event_features = customer_orders_enriched[[
    'customer_id', 'order_id', 'order_line', 'sku_id', 'size_us', 'quantity',
    'timestamp', 'wave_id', 'operator_id', 'abc_code', 'sector',
    'latency_sec', 'residual', 'delta_weight', 'distance_walked'
]]

print("Event-level features created:")
print(event_features.head())
print(f"Shape: {event_features.shape}")

### Order-Level Features

In [ ]:
# Order-level features (aggregated from event-level data)
order_features = customer_orders_enriched.groupby('order_id').agg({
    'customer_id': 'first',
    'timestamp': 'first',  # Order creation time
    'wave_id': 'first',
    'operator_id': 'first',
    'quantity': 'sum',  # Total quantity in order
    'order_size': 'first',  # Number of lines in order
    'delta_weight': 'sum',  # Total weight of order
    'distance_walked': 'sum',  # Total distance for order
    'residual': 'mean',  # Average residual per order
    'latency_sec': 'mean'  # Average latency per order
}).reset_index()

# Add additional calculated features
order_features['order_hour'] = order_features['timestamp'].dt.hour
order_features['order_dow'] = order_features['timestamp'].dt.dayofweek

print("Order-level features created:")
print(order_features.head())
print(f"Shape: {order_features.shape}")

### Station/Operator-Level Features (Hourly)

In [ ]:
# Station/operator-level features (hourly aggregation)
customer_orders_enriched['hour'] = customer_orders_enriched['timestamp'].dt.hour
customer_orders_enriched['date'] = customer_orders_enriched['timestamp'].dt.date

station_hourly_features = customer_orders_enriched.groupby(['operator_id', 'date', 'hour']).agg({
    'order_id': 'nunique',  # Number of orders
    'sku_id': 'count',  # Number of picks
    'quantity': 'sum',  # Total quantity picked
    'delta_weight': 'sum',  # Total weight
    'distance_walked': 'sum',  # Total distance
    'latency_sec': 'mean',  # Average latency
    'residual': 'mean'  # Average residual
}).reset_index()

# Rename columns for clarity
station_hourly_features.rename(columns={
    'order_id': 'orders_count',
    'sku_id': 'picks_count',
    'quantity': 'total_quantity',
    'delta_weight': 'total_weight',
    'distance_walked': 'total_distance',
    'latency_sec': 'avg_latency',
    'residual': 'avg_residual'
}, inplace=True)

print("Station/operator-hourly features created:")
print(station_hourly_features.head())
print(f"Shape: {station_hourly_features.shape}")

## 5. Save Feature Tables

In [ ]:
# Save feature tables to CSV and Parquet formats
event_features.to_csv('../data/event_features.csv', index=False)
event_features.to_parquet('../data/event_features.parquet', index=False)

order_features.to_csv('../data/order_features.csv', index=False)
order_features.to_parquet('../data/order_features.parquet', index=False)

station_hourly_features.to_csv('../data/station_hourly_features.csv', index=False)
station_hourly_features.to_parquet('../data/station_hourly_features.parquet', index=False)

print("Feature tables saved successfully!")
print("- Event-level features: event_features.csv, event_features.parquet")
print("- Order-level features: order_features.csv, order_features.parquet")
print("- Station/hour-level features: station_hourly_features.csv, station_hourly_features.parquet")

## Summary

We have successfully:
1. Standardized column names across all datasets
2. Engineered key features including:
   - delta_weight: Simulated product weight
   - residual: Difference between expected and actual quantities
   - latency_sec: Time between consecutive picks
   - queue_depth: Number of orders in a wave (represented as wave_order_count)
   - distance_walked: Simulated walking distance
3. Created three feature sets:
   - Event-level: Individual picking events
   - Order-level: Aggregated order information
   - Station/hour-level: Operator performance by hour
4. Saved all feature tables in both CSV and Parquet formats for efficient reuse